Here we will evaluate how well the retrieval works....

Generating ground truth data.....

We need questions with known relevant documents.

We generate these with an LLM, asking it to create questions for each symptoms:

In [3]:
# Call the client.

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()


In [4]:
# Generating ground truth data/ create questions for each symptoms....
# choose 100 doc/ generate 2 questions each.

prompt1_template = """
You're an NHS Inform health information expert generating evaluation questions.
For the symptoms below, generate 2 questions that a patient might ask when seeking medical advice.

Return the result as a JSON array with objects containing only 'question' fields.

Chunk id: {chunk_id}
Parent id: {parent_id}
Category: {category}
Section: {section}
Heading: {heading}
url: {url}
Content: {content}

""".strip()


from tqdm.auto import tqdm
import json

with open("../data/nhs-symptom-chunks.json", "r", encoding="utf-8") as f:
    document_set = json.load(f)

print(f'{len(document_set)=}')       # Number of objects


results = []

for row in tqdm(document_set[:100],total=len(document_set[:100])): 
    prompt = prompt1_template.format(**row)
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=[{"role": "user", "content": prompt}]
    )
    questions = json.loads(response.output_text) 

    for q in questions:
        q["chunk_id"] = row["chunk_id"] 
        results.append(q)

print(f'{len(results)=}')



len(document_set)=6855


  0%|          | 0/100 [00:00<?, ?it/s]

len(results)=200


In [5]:
results[:5]

[{'question': 'What are the warning signs of an abdominal aortic aneurysm?',
  'chunk_id': 'abdominal-aortic-aneurysm-1'},
 {'question': 'When should I get urgent medical help for symptoms of an abdominal aortic aneurysm?',
  'chunk_id': 'abdominal-aortic-aneurysm-1'},
 {'question': 'What are the symptoms of an abdominal aortic aneurysm, and how would I know if I need urgent medical help?',
  'chunk_id': 'abdominal-aortic-aneurysm-2'},
 {'question': 'Am I at higher risk of getting an abdominal aortic aneurysm because of my age, smoking, or high blood pressure?',
  'chunk_id': 'abdominal-aortic-aneurysm-2'},
 {'question': 'Could abdominal aortic aneurysm cause pain or a pulsing feeling in my tummy, and should I get it checked?',
  'chunk_id': 'abdominal-aortic-aneurysm-3'}]

In [6]:
# save the questions for later use.
import pandas as pd

df_questions = pd.DataFrame(results)
df_questions.to_csv("../data/ground-truth-retrieval.csv", index=False)

In [7]:
len(df_questions)

200

In [8]:
df_questions.head(5)

,question,chunk_id
0,What are the warning signs of an abdominal aor...,abdominal-aortic-aneurysm-1
1,When should I get urgent medical help for symp...,abdominal-aortic-aneurysm-1
2,What are the symptoms of an abdominal aortic a...,abdominal-aortic-aneurysm-2
3,Am I at higher risk of getting an abdominal ao...,abdominal-aortic-aneurysm-2
4,Could abdominal aortic aneurysm cause pain or ...,abdominal-aortic-aneurysm-3


In [9]:
# Load the ground truth data (questions + id).
import pandas as pd
df_question = pd.read_csv("../data/ground-truth-retrieval.csv")
ground_truth = df_question.to_dict(orient="records")

In [10]:
ground_truth[0]

{'question': 'What are the warning signs of an abdominal aortic aneurysm?',
 'chunk_id': 'abdominal-aortic-aneurysm-1'}